<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name: 
<br>
Github Username: 
<br>
USC ID: 

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [29]:
import glob
import pandas as pd
import os
import numpy as np

Get the AReM Data Set

### (b) Test and Train Data

In [ ]:
col_names = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']

def process_datasets(file_paths):

    file_paths = sorted(file_paths) 
    temp_list = []
    
    for file in file_paths:

        activity = os.path.basename(os.path.dirname(file))
        file_name = os.path.basename(file)

        temp_df = pd.read_csv(file, comment="#", header=None, names=col_names, on_bad_lines='skip')

        temp_df['activity'] = activity
        temp_df['dataset'] = file_name

        temp_list.append(temp_df)

    return pd.concat(temp_list, ignore_index=True)


test_data_files = glob.glob("/Users/bricepatchou/Downloads/AReM/bending1/dataset[1-2].csv")
test_data_files += glob.glob("/Users/bricepatchou/Downloads/AReM/bending2/dataset[1-2].csv")
test_data_files += glob.glob("/Users/bricepatchou/Downloads/AReM/cycling/dataset[1-3].csv")
test_data_files += glob.glob("/Users/bricepatchou/Downloads/AReM/lying/dataset[1-3].csv")
test_data_files += glob.glob("/Users/bricepatchou/Downloads/AReM/sitting/dataset[1-3].csv")
test_data_files += glob.glob("/Users/bricepatchou/Downloads/AReM/standing/dataset[1-3].csv")
test_data_files += glob.glob("/Users/bricepatchou/Downloads/AReM/walking/dataset[1-3].csv")

print(len(test_data_files))

test_data = process_datasets(test_data_files)






training_data_files = glob.glob("/Users/bricepatchou/Downloads/AReM/bending1/dataset[3-7].csv")
training_data_files += glob.glob("/Users/bricepatchou/Downloads/AReM/bending2/dataset[3-6].csv")

other_folders = ["cycling", "lying", "sitting", "standing", "walking"]
base_path = "/Users/bricepatchou/Downloads/AReM"

for folder in other_folders:
    training_data_files += glob.glob(f"{base_path}/{folder}/dataset[4-9].csv")
    training_data_files += glob.glob(f"{base_path}/{folder}/dataset1[0-5].csv")

print(len(training_data_files))

train_data = process_datasets(training_data_files)


print(test_data.head())
print(train_data.head())



19
69
Success: All time-series groups in Test Data have exactly 480 rows.
activity  dataset     
sitting   dataset8.csv    479
dtype: int64
   time  avg_rss12  var_rss12  avg_rss13  var_rss13  avg_rss23  var_rss23  \
0     0      39.25       0.43      22.75       0.43      33.75        1.3   
1   250      39.25       0.43      23.00       0.00      33.00        0.0   
2   500      39.25       0.43      23.25       0.43      33.00        0.0   
3   750      39.50       0.50      23.00       0.71      33.00        0.0   
4  1000      39.50       0.50      24.00       0.00      33.00        0.0   

   activity       dataset  
0  bending1  dataset1.csv  
1  bending1  dataset1.csv  
2  bending1  dataset1.csv  
3  bending1  dataset1.csv  
4  bending1  dataset1.csv  
   time  avg_rss12  var_rss12  avg_rss13  var_rss13  avg_rss23  var_rss23  \
0     0      42.00       0.71      21.25       0.43      30.00       0.00   
1   250      41.50       0.50      20.25       1.48      31.25       1.09  

### (c) Feature Extraction

#### i. Research

#### Common time-domain features include:​
Minimum and Maximum Values: The smallest and largest values in the time series, indicating its range.​
Mean (Average): The central tendency of the data, calculated as the sum of all values divided by the number of observations.​
Median: The middle value when the data is ordered, providing a measure of central tendency less affected by outliers.​
Variance and Standard Deviation: Measures of data dispersion around the mean, indicating the spread or volatility within the series.​
Skewness: Describes the asymmetry of the data distribution.​
Kurtosis: Indicates the "tailedness" of the data distribution, showing the propensity for extreme values.​
Autocorrelation: Measures the correlation between the time series and lagged versions of itself, revealing repeating patterns or periodicity.​
Partial Autocorrelation: Assesses the correlation between the series and its lags, excluding the influence of intermediate lags, helping to identify the direct relationship between observations at different times.​
Zero-Crossing Rate: Counts how often the time series changes sign, useful for understanding the frequency of oscillations.​
Root Mean Square (RMS): Provides a measure of the magnitude of the time series values, emphasizing larger deviations.​
Hjorth Parameters: Comprises Activity (variance of the signal), Mobility (ratio of standard deviation of the first derivative to the signal's standard deviation), and Complexity (comparison of the signal's similarity to a pure sine wave). These parameters are particularly useful in biomedical signal analysis.​

#### ii. Extraction

In [24]:
# col_names = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
# sensor_cols = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']

# all_files = glob.glob("/Users/bricepatchou/Downloads/AReM/bending1/*.csv")
# all_files += glob.glob("/Users/bricepatchou/Downloads/AReM/bending2/*.csv")
# all_files += glob.glob("/Users/bricepatchou/Downloads/AReM/cycling/*.csv")
# all_files += glob.glob("/Users/bricepatchou/Downloads/AReM/lying/*.csv")
# all_files += glob.glob("/Users/bricepatchou/Downloads/AReM/sitting/*.csv")
# all_files += glob.glob("/Users/bricepatchou/Downloads/AReM/standing/*.csv")
# all_files += glob.glob("/Users/bricepatchou/Downloads/AReM/walking/*.csv")



# temp_list = []
# for file in all_files:
#     temp_df = pd.read_csv(file, comment="#", header=None, names=col_names, on_bad_lines='skip')
#     temp_list.append(temp_df)

# df = pd.concat(temp_list, ignore_index=True)

df = pd.concat([train_data, test_data], ignore_index=True)
features_list = []

# df.group_by([['activity', 'dataset']])["avg_rss12","var_rss12","avg_rss13","var_rss13","avg_rss23","var_rss23"].agg(['min','max','mean','median','std',])

for (activity, dataset), group_df in df.groupby(['activity', 'dataset']):

    feature_row = {
        'activity': activity,
        'dataset': dataset
    }

    for i, col in enumerate(sensor_cols, start=1):
        feature_row[f'min{i}'] = group_df[col].min()
        feature_row[f'max{i}'] = group_df[col].max()
        feature_row[f'mean{i}'] = group_df[col].mean()
        feature_row[f'median{i}'] = group_df[col].median()
        feature_row[f'std{i}'] = group_df[col].std()
        feature_row[f'1st_quart{i}'] = group_df[col].quantile(0.25)
        feature_row[f'3rd_quart{i}'] = group_df[col].quantile(0.75)

    features_list.append(feature_row)

final_features_df = pd.DataFrame(features_list)

print(final_features_df.head())

   activity       dataset   min1   max1      mean1  median1      std1  \
0  bending1  dataset1.csv  37.25  45.00  40.624792    40.50  1.476967   
1  bending1  dataset2.csv  38.00  45.67  42.812812    42.50  1.435550   
2  bending1  dataset3.csv  35.00  47.40  43.954500    44.33  1.558835   
3  bending1  dataset4.csv  33.00  47.75  42.179812    43.50  3.670666   
4  bending1  dataset5.csv  33.00  45.75  41.678063    41.75  2.243490   

   1st_quart1  3rd_quart1  min2  ...      std5  1st_quart5  3rd_quart5  min6  \
0       39.25       42.00   0.0  ...  2.188449     33.0000       36.00   0.0   
1       42.00       43.67   0.0  ...  1.995255     32.0000       34.50   0.0   
2       43.00       45.00   0.0  ...  1.999604     35.3625       36.50   0.0   
3       39.15       45.00   0.0  ...  3.849448     30.4575       36.33   0.0   
4       41.33       42.75   0.0  ...  2.411026     28.4575       31.25   0.0   

   max6     mean6  median6      std6  1st_quart6  3rd_quart6  
0  1.92  0.570583

#### iii. Standard Deviation

In [31]:
bootstrap_res= []
B = 1000

for feature in final_features_df.columns:

    if feature == 'activity' or feature == 'dataset':
        continue

    og_std = final_features_df[feature].std(ddof=1)

    boot_stds = []


    for i in range(B):

        resampled_data = final_features_df[feature].sample(n=len(final_features_df), replace=True)
        boot_stds.append(resampled_data.std())


    lower_bound = np.percentile(boot_stds, 5)
    upper_bound = np.percentile(boot_stds, 95)

    bootstrap_res.append({
        'Feature': feature,
        'Original_SD': og_std,
        '90%_CI_Lower': lower_bound,
        '90%_CI_Upper': upper_bound
    })

ci_df = pd.DataFrame(bootstrap_res)


#### iv. Select Features

## 2. ISLR 3.7.4

### (a) Linear Train

### (b) Linear Test

### (c) Not Linear Train

### (d) Not Linear Testing

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 